In [1]:
import pandas as pd

Idea: crear una variable que se llame Madurez Fisiológica donde: 35% HR,  ¾ línea de leche y capa negra para que podamos realizar un primer filtrado que muestre si cumple lo esperado. Tener en cuenta la humedad y el peso de mil de la fecha anterior y siguiente.

In [57]:
# Cargar planillas de datos

excel_TFG_Morales = "DATOS TFG 1819.xlsx"

# Ver las hojas del archivo
xls = pd.ExcelFile(excel_TFG_Morales)
print(xls.sheet_names)


# Cargar datos climáticos
clima = pd.read_excel("CLIMA 1819.xlsx", decimal=",")


['ENS TEMP 201819', 'HUM TEMP', 'ENS TARDIO 201819', 'HUM TARD', 'DATOS HUM', 'Plot Entry', 'Hoja2']


In [58]:
# Guardamos las tablas en DataFrames
ensayo_temp1819 = pd.read_excel(excel_TFG_Morales, sheet_name="ENS TEMP 201819", decimal=",")
ensayo_tardio1819 = pd.read_excel(excel_TFG_Morales, sheet_name="ENS TARDIO 201819", decimal=",")

# Ver las primeras filas
print(ensayo_temp1819.head())
print(ensayo_tardio1819.head())


   PLOT  BLOC  IBLK  ENTRY   ORIGEN PEDIGREE  PTAS         VT  GDU_VT  \
0   410     2    10    167  17.1522    L1037   NaN 2018-12-16  807.75   
1   217     1    10    167  17.1522    L1037   NaN 2018-12-18  835.95   
2   452     2    13     98  17.2886   LP 197   NaN 2018-12-18  835.95   
3   156     1     5    100  17.2890   LP 304   NaN 2018-12-19     NaN   
4   339     2     4    168  17.1536    L2200   NaN 2018-12-19     NaN   

          R1  GDU_R1  ASI  GDU_ASI                   MF                   MC  \
0 2018-12-16  807.75  0.0      NaN  2019-02-18 00:00:00  2019-03-25 00:00:00   
1 2018-12-18  835.95  0.0      NaN  2019-02-18 00:00:00  2019-03-25 00:00:00   
2 2018-12-19     NaN  1.0      NaN  2019-02-18 00:00:00  2019-03-26 00:00:00   
3 2018-12-19     NaN  0.0      NaN                  NaN                  NaN   
4 2018-12-19     NaN  0.0      NaN  2019-02-18 00:00:00  2019-04-01 00:00:00   

     TSG  
0  0.335  
1  0.368  
2    NaN  
3    NaN  
4  0.182  
   PLOT  BLOC 

In [59]:
print(clima.head())

       Fecha  Día  T med  GDU ACUMULADOS FECHA TEMPRANA  \
0 2018-10-01    1  12.45                            NaN   
1 2018-10-02    2   9.40                            NaN   
2 2018-10-03    3  14.15                            NaN   
3 2018-10-04    4  12.85                           4.85   
4 2018-10-05    5  13.20                          10.05   

   GDU ACUMULADOS FECHA TARDÍO  
0                          NaN  
1                          NaN  
2                          NaN  
3                          NaN  
4                          NaN  


In [60]:
# Vemos los nombres de las columnas que tenemos en los dataframes

print(ensayo_temp1819.columns)
print(ensayo_tardio1819.columns)
print(clima.columns)

Index(['PLOT', 'BLOC', 'IBLK', 'ENTRY', 'ORIGEN', 'PEDIGREE', 'PTAS', 'VT',
       'GDU_VT', 'R1', 'GDU_R1', 'ASI', 'GDU_ASI', 'MF', 'MC', 'TSG'],
      dtype='object')
Index(['PLOT', 'BLOC', 'IBLK', 'ENTRY', 'ORIGEN', 'PEDIGREE', 'PTAS', 'VT',
       'GDU_VT', 'R1', 'GDU_R1', 'ASI', 'GDU_ASI', 'MF', 'MC', 'TSG'],
      dtype='object')
Index(['Fecha', 'Día', 'T med', 'GDU ACUMULADOS FECHA TEMPRANA',
       'GDU ACUMULADOS FECHA TARDÍO'],
      dtype='object')


In [61]:
# Dato: función para acomodar nombres de columnas por si tienen caracteres raris
ensayo_temp1819.columns = ensayo_temp1819.columns.str.strip()
ensayo_tardio1819.columns = ensayo_tardio1819.columns.str.strip()

# A las columnas de fechas las convertimos en datetime
ensayo_temp1819["VT"] = pd.to_datetime(ensayo_temp1819["VT"], dayfirst=True)
ensayo_temp1819["R1"] = pd.to_datetime(ensayo_temp1819["R1"], dayfirst=True)
ensayo_tardio1819["R1"] = pd.to_datetime(ensayo_tardio1819["R1"], dayfirst=True)
ensayo_tardio1819["VT"] = pd.to_datetime(ensayo_tardio1819["VT"], dayfirst=True)
clima["Fecha"] = pd.to_datetime(clima["Fecha"], dayfirst=True)

# A las columnas de GDU las convertimos en numéricas teniendo en cuenta la coma decimal
ensayo_temp1819["GDU_VT"] = pd.to_numeric(ensayo_temp1819["GDU_VT"], errors="coerce")
ensayo_temp1819["GDU_R1"] = pd.to_numeric(ensayo_temp1819["GDU_R1"], errors="coerce")
ensayo_tardio1819["GDU_VT"] = pd.to_numeric(ensayo_tardio1819["GDU_VT"], errors="coerce")  
ensayo_tardio1819["GDU_R1"] = pd.to_numeric(ensayo_tardio1819["GDU_R1"], errors="coerce")     

print(ensayo_temp1819.dtypes)
print(ensayo_tardio1819.dtypes)


PLOT                 int64
BLOC                 int64
IBLK                 int64
ENTRY                int64
ORIGEN              object
PEDIGREE            object
PTAS               float64
VT          datetime64[ns]
GDU_VT             float64
R1          datetime64[ns]
GDU_R1             float64
ASI                float64
GDU_ASI            float64
MF                  object
MC                  object
TSG                 object
dtype: object
PLOT                 int64
BLOC                 int64
IBLK                 int64
ENTRY                int64
ORIGEN              object
PEDIGREE            object
PTAS               float64
VT          datetime64[ns]
GDU_VT             float64
R1          datetime64[ns]
GDU_R1             float64
ASI                  int64
GDU_ASI            float64
MF          datetime64[ns]
MC          datetime64[ns]
TSG                float64
dtype: object


Calculamos los GDU teniendo en cuenta que para floración no usamos -8°C. Ya charlamos con Mari de calcular igual los GDU con -8°C

In [54]:

# Asignar los valores de GDU correspondientes buscando la fecha en CLIMA 1819
ensayo_temp1819["GDU_VT"] = ensayo_temp1819["VT"].map(
    clima.set_index("Fecha")["GDU ACUMULADOS FECHA TEMPRANA"]
)
ensayo_temp1819["GDU_R1"] = ensayo_temp1819["R1"].map(
    clima.set_index("Fecha")["GDU ACUMULADOS FECHA TEMPRANA"]
)
ensayo_tardio1819["GDU_VT"] = ensayo_tardio1819["VT"].map(
    clima.set_index("Fecha")["GDU ACUMULADOS FECHA TARDÍO"]
)
ensayo_tardio1819["GDU_R1"] = ensayo_tardio1819["R1"].map(
    clima.set_index("Fecha")["GDU ACUMULADOS FECHA TARDÍO"]
)


In [55]:
# Calculamos GDU_ASI como una diferencia
ensayo_temp1819["GDU_ASI"] = ensayo_temp1819["GDU_R1"] - ensayo_temp1819["GDU_VT"]
ensayo_tardio1819["GDU_ASI"] = ensayo_tardio1819["GDU_R1"] - ensayo_tardio1819["GDU_VT"]    

In [56]:
# Creamos un ExcelWriter y escribir ambas planillas en hojas separadas
with pd.ExcelWriter("ENSAYO_GDUs_calculated.xlsx") as writer:
    ensayo_temp1819.to_excel(writer, sheet_name="ENS_TEMP_1819", index=False)
    ensayo_tardio1819.to_excel(writer, sheet_name="ENS_TARDIO_1819", index=False)

print("✅ Archivo 'ENSAYO_GDUs_calculated.xlsx' creado con dos planillas correctamente.")


✅ Archivo 'ENSAYO_GDUs_calculated.xlsx' creado con dos planillas correctamente.


In [ ]:
# Extraemos pedigríes únicos y los enumeramos para asignarles un ID
genotipos = ensayo_temp1819["PEDIGREE"].dropna().unique()  # Elimina duplicados y NaN
genotipos_df = pd.DataFrame({"Genotipos": genotipos})  # Convierte a DataFrame
genotipos_df.index = genotipos_df.index + 1  # Enumerar desde 1
genotipos_df.index.name = "N°"  # Nombre de la columna índice

# Guardamos en un nuevo Excel
genotipos_df.to_excel("Genotipos_Listados.xlsx", index=True)

print("✅ Archivo 'Genotipos_Listados.xlsx' creado correctamente con los pedigríes enumerados.")


✅ Archivo 'Genotipos_Listados.xlsx' creado correctamente con los pedigríes enumerados.


AHORA OTRO CÓDIGO PARA MANTENER EL FORMATO ORIGINAL DE LA PLANILLA DE EXCEL (con los colores marcados por el codirector)

In [62]:
# Leemos el archivo con openpyxl para no perder formatos
from openpyxl import load_workbook


In [63]:
# Cargamos el archivo original con openpyxl
archivo_original = "ENSAYO_2018-19_GDUs.xlsx" 
wb = load_workbook(archivo_original)

In [64]:
# Cargamos datos climáticos
clima = pd.read_excel("CLIMA 1819.xlsx", decimal=",")

In [65]:
# Listamos todos los nombres de las planillas
xls = pd.ExcelFile(archivo_original)
print(xls.sheet_names)

['ENS TEMP 201819', 'HUM TEMP', 'ENS TARDIO 201819', 'HUM TARD', 'DATOS HUM', 'PlotEntry', 'CALCULO TSG']


In [66]:
# Cargamos cada hoja como un DataFrame
ensayo_temp1819 = pd.read_excel(archivo_original, sheet_name="ENS TEMP 201819")
ensayo_tardio1819 = pd.read_excel(archivo_original, sheet_name="ENS TARDIO 201819")
humtemp = pd.read_excel(archivo_original, sheet_name="HUM TEMP")
humtardio = pd.read_excel(archivo_original, sheet_name="HUM TARD")

In [62]:
# Asignar los valores de GDU_VT y GDU_R1 basados en las fechas de la planilla CLIMA 1819
ensayo_temp1819["GDU_VT"] = ensayo_temp1819["VT"].map(
    clima.set_index("Fecha")["GDU ACUMULADOS FECHA TEMPRANA"]
)
ensayo_temp1819["GDU_R1"] = ensayo_temp1819["R1"].map(
    clima.set_index("Fecha")["GDU ACUMULADOS FECHA TEMPRANA"]
)

ensayo_tardio1819["GDU_VT"] = ensayo_tardio1819["VT"].map(
    clima.set_index("Fecha")["GDU ACUMULADOS FECHA TARDÍO"]
)
ensayo_tardio1819["GDU_R1"] = ensayo_tardio1819["R1"].map(
    clima.set_index("Fecha")["GDU ACUMULADOS FECHA TARDÍO"]
)

In [63]:
print(ensayo_tardio1819.head())


    PLOT  BLOC  IBLK  ENTRY         ORIGEN                       PEDIGREE  \
0  101.0   1.0   1.0   60.0        17.2804   [BSGA] 04.4060-1.1.1.1.1.#.#   
1  102.0   1.0   1.0   85.0        17.2860     [BS29 ] 04.696-1.1.1.1.1.#   
2  103.0   1.0   1.0  129.0  17.2948//BEBA                         LP 611   
3  104.0   1.0   1.0   64.0        17.2812   [BSGA] 04.4092-1.1.1.1.1.#.#   
4  105.0   1.0   1.0   95.0        17.2880  [FESTIVAL] 04.785-1.1.1.1.1.#   

   PTAS         VT   GDU_VT         R1  ...         MF         MC    TSG  \
0   NaN 2019-02-25  1182.60 2019-02-27  ... 2019-05-10 2019-08-16  0.193   
1   NaN 2019-02-25  1182.60 2019-02-25  ... 2019-05-30 2019-08-24  0.275   
2   NaN 2019-02-27  1198.75 2019-03-01  ...        NaT        NaT    NaN   
3   NaN 2019-02-26  1188.80 2019-02-27  ...        NaT        NaT  0.107   
4   NaN 2019-02-28  1210.90 2019-02-28  ... 2019-05-13 2019-08-15  0.264   

  Tallo Roya  MRCV  TIZON  BACT  SG  OBSERV  
0   NaN  NaN   NaN    NaN   NaN Na

In [64]:
ensayo_temp1819["GDU_ASI"] = ensayo_temp1819["GDU_R1"] - ensayo_temp1819["GDU_VT"]
ensayo_tardio1819["GDU_ASI"] = ensayo_tardio1819["GDU_R1"] - ensayo_tardio1819["GDU_VT"]

In [65]:

# Reemplazar los valores en la hoja original sin modificar el formato
def reemplazar_columna(wb, sheet_name, df, col_name, col_letter):
    """Reemplaza los valores de una columna en el archivo original sin perder formato"""
    ws = wb[sheet_name]  # Acceder a la sheet del archivo original
    col_idx = df.columns.get_loc(col_name) + 1  # Encontrar la posición de la columna en el DataFrame
    for i, valor in enumerate(df[col_name], start=2):  # Asumimos que los datos empiezan en la fila 2
        ws[f"{col_letter}{i}"] = valor  # Reemplazamos el valor en la celda sin cambiar formato

# Aplicar reemplazo en ambas hojas
reemplazar_columna(wb, "ENS TEMP 201819", ensayo_temp1819, "GDU_VT", "I")
reemplazar_columna(wb, "ENS TEMP 201819", ensayo_temp1819, "GDU_R1", "K")
reemplazar_columna(wb, "ENS TEMP 201819", ensayo_temp1819, "GDU_ASI", "L")

reemplazar_columna(wb, "ENS TARDIO 201819", ensayo_tardio1819, "GDU_VT", "I")
reemplazar_columna(wb, "ENS TARDIO 201819", ensayo_tardio1819, "GDU_R1", "K")
reemplazar_columna(wb, "ENS TARDIO 201819", ensayo_tardio1819, "GDU_ASI", "L")



In [66]:
# Guardar el archivo modificado manteniendo el formato original
wb.save("ENSAYO_2018-19_GDUs.xlsx")

print("✅ Archivo actualizado sin perder formato: 'ENSAYO_2018-19_GDUs.xlsx'")

✅ Archivo actualizado sin perder formato: 'ENSAYO_2018-19_GDUs.xlsx'


VARIABLE MADUREZ FISIOLÓGICA - CRITERIOS (curva sigmoidea, no se linealiza): 
- HR entre 32 y 40%
- Peso seco máximo dentro del plot

VARIABLE MADUREZ COMERCIAL - CRITERIOS (linealizamos 14.5% HR):
- rango HR entre 10 y 16%
- Peso seco no tenemos en cuenta ya que es constante?
- Se selecciona la primera fecha registrada en ese rango para cada Plot (la primera vez que se alcanza la MC)

In [75]:

# Cargamos nuevamente datos
df = pd.read_excel("ENSAYO_2018-19_GDUs.xlsx", sheet_name= "HUM TARD", decimal=",")

# Convertimos "HUM" a numérico por si hay valores no reconocidos como número
df["HUM X"] = pd.to_numeric(df["HUM"], errors="coerce")

# ------ MADUREZ FISIOLÓGICA -----
# Filtramos por humedad entre 32% y 40% para MF
df_mf = df[(df["HM X"] >= 32) & (df["HM X"] <= 40)]

# ------ MADUREZ COMERCIAL -----
# Filtramos por humedad entre 10% y 16% para MC
df_mc = df[(df["HM X"] >= 10) & (df["HM X"] <= 16)]

# Encontramos el máximo Peso Seco dentro del rango para cada Plot (único valor MF por Plot)
df_mf = df_mf.loc[df_mf.groupby("Plot")["Peso Seco"].idxmax()]

# Encontramos el primer registro dentro del rango de MC para cada Plot (único valor MC por Plot)
df_mc = df_mc.loc[df_mc.groupby("Plot")["Fecha"].idxmin()]

# Inicializamos las columnas como False para asegurarnos de que solo se marquen las correctas
df["Madurez_Fisiologica"] = False
df["Madurez_Comercial"] = False

# Asignamos True solo a las filas seleccionadas
df.loc[df_mf.index, "Madurez_Fisiologica"] = True
df.loc[df_mc.index, "Madurez_Comercial"] = True

# Guardamos el DataFrame actualizado
df.to_excel("DATOS_TFG_CORREGIDO.xlsx", index=False)

In [76]:
# Guardar el resultado en un nuevo archivo
df.to_excel("ENSAYO_TARD_MF_MC.xlsx", index=False)

print("Automatización del cálculo de Madurez Fisiológica completado.")

Automatización del cálculo de Madurez Fisiológica completado.


In [9]:
import pandas as pd
import numpy as np

df = pd.read_excel("ENSAYO_2018-2019_GoogleSheets.xlsx", sheet_name="ENS TEMP 201819", decimal=",")
ensayo_tardio1819 = pd.read_excel("ENSAYO_2018-2019_GoogleSheets.xlsx", sheet_name="ENS TARDIO 201819", decimal=",")

def clasificar_filas(df):
    
    df = df.sort_values("Fecha")
    df["Esperado"] = True
    
    for i in range(1, len(df)):
        
        hum_actual = df.iloc[i]["HUM"]
        hum_prev = df.iloc[i-1]["HUM"]
        
        ps_actual = df.iloc[i]["Peso_Seco"]
        ps_prev = df.iloc[i-1]["Peso_Seco"]
        
        if hum_actual > hum_prev or ps_actual < ps_prev:
            df.iloc[i, df.columns.get_loc("Esperado")] = False
            
    return df


def calcular_MF(df):
    
    max_ps = df["Peso_Seco"].max()
    
    candidatos = df[
        (df["HUM"] >= 32) &
        (df["HUM"] <= 40) &
        (df["Peso_Seco"] == max_ps)
    ]
    
    if len(candidatos) > 0:
        return candidatos.iloc[0]["Fecha"], True
    
    return None, False


def extrapolar_MC(df):

    df = df.sort_values("Fecha")

    x = (df["Fecha"] - df["Fecha"].min()).dt.days
    y = df["HUM"]

    coef = np.polyfit(x, y, 1)

    a, b = coef
    
    x_target = (14.5 - b) / a
    
    fecha_mc = df["Fecha"].min() + pd.Timedelta(days=x_target)

    return fecha_mc


def calcular_MC(df):

    rango = df[(df["HUM"] >= 10) & (df["HUM"] <= 16)]

    if len(rango) > 0:
        return rango.iloc[0]["Fecha"], "observada"

    ultima = df.iloc[-1]

    if ultima["HUM"] > 14.5:
        return extrapolar_MC(df), "extrapolada"

    return None, "duda"


def analizar_plot(df_plot):

    df_plot = clasificar_filas(df_plot)

    df_validas = df_plot[df_plot["Esperado"] == True]

    if len(df_validas) == 0:
        return {"decision": "Se descarta"}

    MF, MF_valida = calcular_MF(df_validas)

    if MF_valida == False:
        return {"decision": "Se descarta"}

    MC, tipo = calcular_MC(df_validas)

    if MC is None:
        return {"decision": "Duda", "MF": MF}

    return {
        "decision": "Plot válido",
        "MF": MF,
        "MC": MC,
        "tipo_MC": tipo
    }

In [12]:
df = pd.read_excel("ENSAYO_2018-2019_GoogleSheets.xlsx", sheet_name="HUM TEMP", decimal=",")

# Compatibilizar nombres para analizar_plot (usa HUM y Peso_Seco)
df = df.rename(columns={"HUM X": "HUM", "Peso Seco": "Peso_Seco"})

# Convertir HUM y Peso_Seco a numérico para evitar errores de comparación con strings
df["HUM"] = pd.to_numeric(df["HUM"], errors="coerce")
df["Peso_Seco"] = pd.to_numeric(df["Peso_Seco"], errors="coerce")

resultados = []
# Add debugging to identify problematic plots
def analizar_plot_debug(df_plot, plot_id):
    try:
        result = analizar_plot(df_plot)
        return result
    except np.linalg.LinAlgError as e:
        print(f"⚠️ LinAlgError in Plot {plot_id}: {e}")
        print(f"Data shape: {df_plot.shape}")
        print(f"Humidity range: {df_plot['HUM'].min()} - {df_plot['HUM'].max()}")
        print(f"Date range: {df_plot['Fecha'].min()} - {df_plot['Fecha'].max()}")
        return {"decision": "Error_numerico", "error": str(e)}

# Use in your main loop
for plot, grupo in df.groupby("Plot", dropna=True):
    r = analizar_plot_debug(grupo, plot)
for plot, grupo in df.groupby("Plot", dropna=True):
        def extrapolar_MC(df):
        df = df.sort_values("Fecha")
        
        # Check if we have enough data points
        if len(df) < 2:
            return None
        
        x = (df["Fecha"] - df["Fecha"].min()).dt.days
        y = df["HUM"]
        
        # Remove NaN values
        mask = ~(np.isnan(x) | np.isnan(y))
        x_clean = x[mask]
        y_clean = y[mask]
        
        # Check for valid data after cleaning
        if len(x_clean) < 2:
            return None
        
        # Check for identical x values (no time variation)
        if np.all(x_clean == x_clean[0]):
            return None
            
        # Check for identical y values (no humidity va        def calcular_MC(df):
            # First try observed data
            rango = df[(df["HUM"] >= 10) & (df["HUM"] <= 16)]
            
            if len(rango) > 0:
                return rango.iloc[0]["Fecha"], "observada"
            
            # Try extrapolation only if we have sufficient data
            if len(df) < 2:
                return None, "insuficientes_datos"
                
            ultima = df.iloc[-1]
            
            if ultima["HUM"] > 14.5:
                fecha_extrapolada = extrapolar_MC(df)
                if fecha_extrapolada is not None:
                    return fecha_extrapolada, "extrapolada"
                else:
                    return None, "extrapolacion_fallida"
            
            return None, "duda"

    # Convertir plot a int de forma segura para evitar errores de tipado (Scalar -> ConvertibleToInt)
    if isinstance(plot, (int, np.integer)):
        plot_id = int(plot)
    elif isinstance(plot, (float, np.floating)) and np.isfinite(plot):
        plot_id = int(plot)
    elif isinstance(plot, str):
        plot_num = pd.to_numeric(plot, errors="coerce")
        if pd.isna(plot_num):
            continue
        plot_id = int(plot_num)
    else:
        continue

    resultados.append({"Plot": plot_id, **r})

df_resultados = pd.DataFrame(resultados)

IndentationError: unindent does not match any outer indentation level (<string>, line 73)

In [5]:
# Crear columnas adicionales para identificar el ensayo
ensayo_temp1819["Año"] = 2018
ensayo_temp1819["Localidad"] = "Pergamino"
ensayo_temp1819["Ensayo"] = "Temprano"

ensayo_tardio1819["Año"] = 2018
ensayo_tardio1819["Localidad"] = "Pergamino"
ensayo_tardio1819["Ensayo"] = "Tardío"

NameError: name 'ensayo_temp1819' is not defined

In [ ]:
# Seleccionar columnas de interés y unificar
columnas_finales = ["Año", "Localidad", "Ensayo", "PEDIGREE", "GDU_VT", "GDU_R1", "GDU_ASI", "TSG", "MC", "MF"]

df_final = pd.concat([ensayo_temp1819[columnas_finales], ensayo_tardio1819[columnas_finales]], ignore_index=True)

# Guardar el bosquejo en una nueva planilla
df_final.to_excel("Bosquejo_Base_Datos.xlsx", index=False)

print("¡Bosquejo generado con éxito!")

¡Bosquejo generado con éxito!


Mismo código pero manteniendo el formato de la planilla de excel

In [ ]:
# Cargamos los datos con el formato original
df = pd.read_excel("ENSAYO_2018-19_GDUs.xlsx", sheet_name="HUM TEMP", decimal=",")

# Convertimos "HUM" a numérico por si hay valores no reconocidos como número
df["HUM"] = pd.to_numeric(df["HUM"], errors="coerce")

# ------ MADUREZ FISIOLÓGICA -----
# Filtramos por humedad entre 32% y 40% para MF
df_mf = df[(df["HUM"] >= 32) & (df["HUM"] <= 40)]

# ------ MADUREZ COMERCIAL -----
# Filtramos por humedad entre 10% y 16% para MC
df_mc = df[(df["HUM"] >= 10) & (df["HUM"] <= 16)]

# Encontramos el máximo Peso Seco dentro del rango para cada Plot (único valor MF por Plot)
df_mf = df_mf.loc[df_mf.groupby("Plot")["Peso Seco"].idxmax()]

# Encontramos el primer registro dentro del rango de MC para cada Plot (único valor MC por Plot)
df_mc = df_mc.loc[df_mc.groupby("Plot")["Fecha"].idxmin()]

# Creamos las columnas y asignamos el valor False por defecto para mantener el formato
df["Madurez_Fisiologica"] = False
df["Madurez_Comercial"] = False

# Asignamos True solo a las filas correspondientes
df.loc[df_mf.index, "Madurez_Fisiologica"] = True
df.loc[df_mc.index, "Madurez_Comercial"] = True

In [ ]:
# Guardamos el DataFrame actualizado con formato original
with pd.ExcelWriter("ENSAYO_2018-19_GDUs_CORREGIDO.xlsx", engine="xlsxwriter") as writer:
    df.to_excel(writer, sheet_name="ENS_TEMP_201819", index=False)

print("Automatización del cálculo de Madurez Fisiológica y Comercial completada.")


Automatización del cálculo de Madurez Fisiológica y Comercial completada.


In [ ]:
# ------- BOSQUEJO DE LA BASE DE DATOS FINAL --------
# Cargar las dos hojas de Excel
df_temp = pd.read_excel("ENSAYO_2018-19_GDUs.xlsx", sheet_name="ENS TEMP 201819", decimal=",")
df_tardio = pd.read_excel("ENSAYO_2018-19_GDUs.xlsx", sheet_name="ENS TARDIO 201819", decimal=",")

# Agregar información del ensayo
df_temp["Año"] = 2018
df_temp["Localidad"] = "Pergamino"
df_temp["Ensayo"] = "Temprano"

df_tardio["Año"] = 2018
df_tardio["Localidad"] = "Pergamino"
df_tardio["Ensayo"] = "Tardío"

# Verificamos que las columnas necesarias existan en ambas bases
columnas_necesarias = ["PLOT", "PEDIGREE", "GDU_VT", "GDU_R1", "GDU_ASI", "TSG"]
columnas_faltantes_temp = [col for col in columnas_necesarias if col not in df_temp.columns]
columnas_faltantes_tardio = [col for col in columnas_necesarias if col not in df_tardio.columns]

if columnas_faltantes_temp or columnas_faltantes_tardio:
    print(f"⚠️ Faltan columnas en ENS_TEMP_201819: {columnas_faltantes_temp}")
    print(f"⚠️ Faltan columnas en ENS_TARDIO_201819: {columnas_faltantes_tardio}")
    raise ValueError("Faltan columnas necesarias en los datos, revisar nombres.")

# Seleccionar solo las columnas necesarias
columnas_finales = ["Año", "Localidad", "Ensayo"] + columnas_necesarias
df_temp = df_temp[columnas_finales]
df_tardio = df_tardio[columnas_finales]

# Unificamos ambas bases de datos
df_final = pd.concat([df_temp, df_tardio], ignore_index=True)

# Guardamos el resultado en una nueva planilla
with pd.ExcelWriter("Bosquejo_Base_Datos.xlsx", engine="xlsxwriter") as writer:
    df_final.to_excel(writer, sheet_name="Datos Finales", index=False)

print("✅ ¡Bosquejo generado con éxito!")

✅ ¡Bosquejo generado con éxito!


Identificamos las parcelas con el comportamiento esperado e inesperado para observar con mayor detenimiento aquellos plots con comportamiento inesperado, de manera de eliminar el plot o determinar manualmente la MF y MC si lo requiere. 
Criterios para la clasificación:

"Esperado":
- La humedad (HUM) debe disminuir en sucesivos muestreos.
- El peso seco (Peso Seco) debe aumentar en sucesivos muestreos.

"No esperado":

- Si alguna fila rompe la tendencia de humedad creciente o peso seco decreciente, se marcará todo el Plot como "No esperado"... me parece medio drástico, se marcarán esas filas que tienen valores extraños como "No esperado" ya que seguro el dato estuvo mal tomado y luego se descartan esos datos, pero solo se descarta el plot si todas las filas para ese plot son "No esperado" o si no tenemos forma de calcular la madurez fisiológica. 

In [ ]:
# Cargar la base de datos
archivo = "ENSAYO_2018-19_GDUs.xlsx"
df_gdu_temp = pd.read_excel(archivo, sheet_name="HUM TEMP", decimal=",")
df_gdu_tardio = pd.read_excel(archivo, sheet_name="HUM TARD", decimal=",")

In [ ]:
print(df_gdu_temp.head())

   Nº Muestra  Lata  Plot  Muestreo      Fecha  Peso Tara  Peso Húmedo  \
0         393    43   101         2 2019-02-21       5211       6772.0   
1         394    44   101         2 2019-02-21       5218       6781.0   
2        1047     1   101         4 2019-02-28       5301       7573.0   
3        1048     2   101         4 2019-02-28       5190       7578.0   
4        1697     1   101         6 2019-03-11       5301       7320.0   

   Peso Seco  Chala        HUM      HUM X        DE  Nº muestreo  TSG  \
0     6127.0   10.0  70.414847  67.211621  4.530046          2.0  NaN   
1     6171.0    NaN  64.008395        NaN       NaN          NaN  NaN   
2     6914.0    5.0  40.855549  40.995150  0.197426          NaN  NaN   
3     6882.0    NaN  41.134752        NaN       NaN          NaN  NaN   
4     6881.0    5.0  27.784810  27.462934  0.455202          NaN  NaN   

   PESO GRANO  
0        91.6  
1        95.3  
2       161.3  
3       169.2  
4       158.0  


In [ ]:
# Convertir a numérico para evitar problemas con valores no reconocidos
df_gdu_temp["HUM"] = pd.to_numeric(df_gdu_temp["HUM"], errors="coerce")
df_gdu_temp["Peso Seco"] = pd.to_numeric(df_gdu_temp["Peso Seco"], errors="coerce")
df_gdu_tardio["HUM"] = pd.to_numeric(df_gdu_temp["HUM"], errors="coerce")
df_gdu_tardio["Peso Seco"] = pd.to_numeric(df_gdu_temp["Peso Seco"], errors="coerce")

In [ ]:
# Corregir posibles espacios en los nombres de columnas
df_gdu_temp.columns = df_gdu_temp.columns.str.strip()
df_gdu_tardio.columns = df_gdu_tardio.columns.str.strip()


In [ ]:
# Ordenamos los datos por Plot y Fecha
df_gdu_temp = df_gdu_temp.sort_values(by=["Plot", "Fecha"])
df_gdu_tardio = df_gdu_tardio.sort_values(by=["Plot", "Fecha"])

# Función para evaluar comportamiento por Plot
def evaluar_plot(grupo):
    humedad_decreciente = grupo["HUM"].is_monotonic_decreasing
    peso_creciente = grupo["Peso Seco"].is_monotonic_increasing
    return "Esperado" if humedad_decreciente and peso_creciente else "No esperado"

# Aplicamos la evaluación a cada Plot
df_gdu_temp["Estado_Parcela"] = df_gdu_temp.groupby("Plot").apply(evaluar_plot).reset_index(level=0, drop=True)
df_gdu_tardio["Estado_Parcela"] = df_gdu_tardio.groupby("Plot").apply(evaluar_plot).reset_index(level=0, drop=True)

# Guardamos el resultado manteniendo el formato original
with pd.ExcelWriter("ENSAYO_2018-19_LOGICA.xlsx", engine="xlsxwriter") as writer:
    df_gdu_temp.to_excel(writer, sheet_name="HUM TEMP", index=False)
    df_gdu_tardio.to_excel(writer, sheet_name="HUM TARD", index=False)

print("✅ Clasificación de parcelas completada con éxito.")


C:\Users\valen\AppData\Local\Temp\ipykernel_19800\4272046496.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_gdu_temp["Estado_Parcela"] = df_gdu_temp.groupby("Plot").apply(evaluar_plot).reset_index(level=0, drop=True)
C:\Users\valen\AppData\Local\Temp\ipykernel_19800\4272046496.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_gdu_tardio["Estado_Parcela"] = df_gdu_tardio.groupby("Plot").apply(

✅ Clasificación de parcelas completada con éxito.


In [ ]:
# Limpiar nombres de columnas
df_gdu_temp.columns = df_gdu_temp.columns.str.strip()
df_gdu_tardio.columns = df_gdu_tardio.columns.str.strip()

# Verificar que la columna 'HUM' exista en ambos DataFrames
if "HUM" not in df_gdu_temp.columns or "HUM" not in df_gdu_tardio.columns:
    raise KeyError("La columna 'HUM' no se encuentra en uno o ambos DataFrames. Verifique los nombres de las columnas.")

# Ordenar para evaluar correctamente el comportamiento por fecha
df_gdu_temp = df_gdu_temp.sort_values(by=["Plot", "Fecha", "Muestreo", "Lata"])
df_gdu_tardio = df_gdu_tardio.sort_values(by=["Plot", "Fecha", "Muestreo", "Lata"])

# Función para evaluar comportamiento en cada parcela
def evaluar_plot(grupo):
    humedad_decreciente = grupo["HUM"].is_monotonic_decreasing
    peso_creciente = grupo["Peso Seco"].is_monotonic_increasing
    return "Esperado" if humedad_decreciente and peso_creciente else "No esperado"

# Agrupar por `Plot`, `Fecha`, `Muestreo`, `Lata`
df_gdu_temp["Estado_Parcela"] = df_gdu_temp.groupby(["Plot", "Fecha", "Muestreo", "Lata"]).apply(
    lambda grupo: evaluar_plot(grupo)
).reset_index(level=[0, 1, 2, 3], drop=True)

df_gdu_tardio["Estado_Parcela"] = df_gdu_tardio.groupby(["Plot", "Fecha", "Muestreo", "Lata"]).apply(
    lambda grupo: evaluar_plot(grupo)
).reset_index(level=[0, 1, 2, 3], drop=True)

# Guardar en un nuevo archivo
with pd.ExcelWriter("ENSAYO_2018-19_LOGICA.xlsx", engine="xlsxwriter") as writer:
    df_gdu_temp.to_excel(writer, sheet_name="HUM TEMP", index=False)
    df_gdu_tardio.to_excel(writer, sheet_name="HUM TARD", index=False)

print("✅ Clasificación corregida y aplicada a todas las filas correctamente.")


C:\Users\valen\AppData\Local\Temp\ipykernel_19800\187752213.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_gdu_temp["Estado_Parcela"] = df_gdu_temp.groupby(["Plot", "Fecha", "Muestreo", "Lata"]).apply(
C:\Users\valen\AppData\Local\Temp\ipykernel_19800\187752213.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_gdu_tardio["Estado_Parcela"] = df_gdu_tardio.groupby(["Plot", "Fecha", "Muestreo", 

✅ Clasificación corregida y aplicada a todas las filas correctamente.


AUTOMATIZACIÓN CÁLCULO TSG

In [ ]:
import pandas as pd

# Cargar las planillas
hum_tard = pd.read_excel("ENSAYO_2018-2019_Tornello.xlsx", sheet_name="HUM TARD", decimal=",")
ens_tardio = pd.read_excel("ENSAYO_2018-2019_Tornello.xlsx", sheet_name="ENS TARDIO 201819", decimal=",")

In [ ]:
# Dato: función para acomodar nombres de columnas por si tienen caracteres raris
hum_tard.columns = hum_tard.columns.str.strip()
ens_tardio.columns = ens_tardio.columns.str.strip()

# A las columnas de fechas las convertimos en datetime
ens_tardio["VT"] = pd.to_datetime(ens_tardio["VT"], dayfirst=True)
ens_tardio["R1"] = pd.to_datetime(ens_tardio["R1"], dayfirst=True)
ens_tardio["R1"] = pd.to_datetime(ens_tardio["R1"], dayfirst=True)
ens_tardio["MC"] = pd.to_datetime(ens_tardio["MC"], dayfirst=True)
ens_tardio["MF"] = pd.to_datetime(ens_tardio["MF"], dayfirst=True)
hum_tard["Fecha"] = pd.to_datetime(hum_tard["Fecha"], dayfirst=True)

# Columnas tipo texto

ens_tardio["PEDIGREE"] = ens_tardio["PEDIGREE"].astype(str)
hum_tard["MF"] = hum_tard["MF"].astype(str)     
hum_tard["MC"] = hum_tard["MC"].astype(str)
hum_tard["Comportamiento Peso"] = hum_tard["Comportamiento HM X"].astype(str)
hum_tard["Decisión"] = hum_tard["Decisión"].astype(str)

# A las columnas TSG las convertimos en numéricas teniendo en cuenta la coma decimal
ens_tardio["TSG"] = pd.to_numeric(ens_tardio["TSG"], errors="coerce")
hum_tard["TSG"] = pd.to_numeric(hum_tard["TSG"], errors="coerce")
ens_tardio["PLOT"] = pd.to_numeric(ens_tardio["PLOT"], errors="coerce")
hum_tard["PLOT"] = pd.to_numeric(hum_tard["PLOT"], errors="coerce")

print(ens_tardio.dtypes)
print(hum_tard.dtypes)


PLOT                 int64
BLOC                 int64
IBLK                 int64
ENTRY                int64
ORIGEN              object
PEDIGREE            object
PTAS               float64
VT          datetime64[ns]
GDU_VT             float64
R1          datetime64[ns]
GDU_R1             float64
ASI                float64
GDU_ASI            float64
MF          datetime64[ns]
MC          datetime64[ns]
TSG                float64
Tallo              float64
Roya               float64
MRCV               float64
TIZON              float64
BACT               float64
SG                 float64
OBSERV             float64
Decisión            object
dtype: object
Nº Muestra                      int64
Lata                            int64
PLOT                            int64
Muestreo                        int64
Fecha                  datetime64[ns]
Peso Tara                       int64
Peso Húmedo                    object
Peso Seco                     float64
Chala                          obj

In [ ]:
print(ens_tardio['PLOT'].dtype)
print(hum_tard['PLOT'].dtype)	

int64
int64


In [ ]:

# Función para detectar MF, MC y calcular TSG NO FUNCIONAAA AVISO
def detectar_mf_mc(df):
    mf, mc, tsg = None, None, None
    fila_mc = None
    
    for idx, row in df.iterrows():
        if 32 <= row['HUM'] <= 40 and (mf is None or row['Peso Seco'] > df.loc[df['Fecha'] == mf, 'Peso Seco'].values[0]):
            mf = row['Fecha']
        
        if row['MC'] == 'VERDADERO':
            mc = row['Fecha']
            fila_mc = idx  # Guardamos la fila donde MC es VERDADERO
    
    # Extrapolación de MC a 14.5%
    if mc and df['HUM'].max() >= 16:
        humedad_inicial = df.loc[df['Fecha'] == mc, 'HUM'].values[0]
        if humedad_inicial > 14.5:
            dias_extrapolados = (humedad_inicial - 14.5) / ((df['HUM'].shift(1) - df['HUM']).mean())
            mc = mc - pd.Timedelta(days=dias_extrapolados)
    
    if mf and mc:
        hum_mf = df.loc[df['Fecha'] == mf, 'HUM'].values
        hum_mc = df.loc[df['Fecha'] == mc, 'HUM'].values
        
        if len(hum_mf) > 0 and len(hum_mc) > 0:
            hum_mf = hum_mf[0]
            hum_mc = hum_mc[0]
            tsg = (hum_mf - hum_mc) / (mc - mf).days if (mc - mf).days > 0 else None
    
    return mf, mc, tsg, fila_mc

# Aplicar detección por plot y asignar TSG en la fila correcta
for plot in hum_tard['PLOT'].unique():
    subset = hum_tard[hum_tard['PLOT'] == plot]
    mf, mc, tsg, fila_mc = detectar_mf_mc(subset)
    
    if fila_mc is not None:
        hum_tard.at[fila_mc, 'TSG'] = tsg

# Guardar la planilla actualizada con TSG en la fila de MC VERDADERO
hum_tard.to_excel("HUM_TARD_ACTUALIZADO.xlsx", index=False)

# Merge con ENS TARDIO
ens_tardio = ens_tardio.merge(hum_tard[['PLOT', 'MF', 'MC', 'TSG']], on='PLOT', how='left', suffixes=('', '_hum'))

# Rename columns to avoid conflicts
ens_tardio.rename(columns={'MF': 'MF_ens', 'MC': 'MC_ens', 'TSG': 'TSG_ens'}, inplace=True)

ens_tardio.to_excel("ENS_TARDIO_ACTUALIZA2.xlsx", index=False)


In [ ]:
# Función para detectar MF, MC y calcular TSG
def detectar_mf_mc(df):
    mf, mc, tsg = None, None, None
    fila_mc = None
    plot_valido = False
    
    for idx, row in df.iterrows():
        if 32 <= row['HUM'] <= 40 and (mf is None or row['Peso Seco'] > df.loc[df['Fecha'] == mf, 'Peso Seco'].values[0]):
            mf = row['Fecha']
            plot_valido = True  # Se puede detectar MF, entonces el plot es válido
        
        if row['MC'] == 'VERDADERO':
            mc = row['Fecha']
            fila_mc = idx  # Guardamos la fila donde MC es VERDADERO
    
    if plot_valido and mf and mc:
        hum_mf = df.loc[df['Fecha'] == mf, 'HUM'].values
        hum_mc = df.loc[df['Fecha'] == mc, 'HUM'].values
        
        if len(hum_mf) > 0 and len(hum_mc) > 0:
            hum_mf = hum_mf[0]
            hum_mc = hum_mc[0]
            tsg = (hum_mf - hum_mc) / (mc - mf).days if (mc - mf).days > 0 else None
    
    return mf, mc, tsg, fila_mc, plot_valido

# Aplicar detección por plot y asignar TSG en la fila correcta
for plot in hum_tard['PLOT'].unique():
    subset = hum_tard[hum_tard['PLOT'] == plot]
    mf, mc, tsg, fila_mc, plot_valido = detectar_mf_mc(subset)
    
    if fila_mc is not None:
        hum_tard.at[fila_mc, 'TSG'] = tsg
    
    hum_tard.loc[hum_tard['PLOT'] == plot, 'Decisión'] = 'Plot válido' if plot_valido else 'Se descarta'

# Guardar la planilla actualizada con TSG en la fila de MC VERDADERO
hum_tard.to_excel("HUM_TARD_ACTUALIZADO.xlsx", index=False)

# Merge con ENS TARDIO solo para plots válidos
ens_tardio = ens_tardio.merge(hum_tard[['PLOT', 'MF', 'MC', 'TSG', 'Decisión']], on='PLOT', how='left')
ens_tardio.to_excel("ENS_TARDIO_ACTUALIZADO.xlsx", index=False)


In [ ]:
print(ens_tardio.dtypes)

PLOT                 int32
BLOC                 int64
IBLK                 int64
ENTRY                int64
ORIGEN              object
PEDIGREE            object
PTAS               float64
VT          datetime64[ns]
GDU_VT             float64
R1          datetime64[ns]
GDU_R1             float64
ASI                float64
GDU_ASI            float64
MF          datetime64[ns]
MC          datetime64[ns]
TSG                float64
Tallo              float64
Roya               float64
MRCV               float64
TIZON              float64
BACT               float64
SG                 float64
OBSERV             float64
Decisión            object
dtype: object


In [ ]:
def evaluar_comportamiento(df):
    df = df.sort_values(by='Fecha')  # Ordenar por fecha para evaluar tendencias
    df['Comportamiento HM X'] = 'Esperado'
    df['Comportamiento Peso'] = 'Esperado'
    
    for i in range(1, len(df)):
        if df.iloc[i]['HUM'] > df.iloc[i-1]['HUM']:
            df.at[df.index[i], 'Comportamiento HM X'] = 'No Esperado'
        if df.iloc[i]['Peso Seco'] < df.iloc[i-1]['Peso Seco']:
            df.at[df.index[i], 'Comportamiento Peso'] = 'No Esperado'
    
    return df

def determinar_mf(df):
    df_mf = df[(df['HUM'] >= 32) & (df['HUM'] <= 40)]
    if not df_mf.empty:
        mf = df_mf.loc[df_mf['Peso Seco'].idxmax(), 'Fecha']  # Mayor peso seco dentro del rango
        return mf
    return None

def calcular_tsg(df, mf):
    df_mc = df[(df['HUM'] >= 10) & (df['HUM'] <= 16)]
    if not df_mc.empty:
        mc = df_mc.iloc[0]['Fecha']  # Primera fecha con humedad en rango MC
        hum_mf = df.loc[df['Fecha'] == mf, 'HUM'].values[0]
        hum_mc = df.loc[df['Fecha'] == mc, 'HUM'].values[0]
        tsg = (hum_mf - hum_mc) / (mc - mf).days if (mc - mf).days > 0 else None
        return mc, tsg
    return None, None

def procesar_plots(df):
    # Ensure column names are stripped of leading/trailing spaces
    df.columns = df.columns.astype(str).str.strip()
    
    if 'PLOT' not in df.columns:
        raise KeyError("The column 'PLOT' is missing from the DataFrame. Please check the input data.")
    
    resultados = []
    for plot in df['PLOT'].unique():
        subset = df[df['PLOT'] == plot].copy()
        subset = evaluar_comportamiento(subset)
        mf = determinar_mf(subset)
        
        if mf:
            mc, tsg = calcular_tsg(subset, mf)
            decision = 'Plot Válido'
        else:
            mc, tsg, decision = None, None, 'Se descarta'
        
        df.loc[df['PLOT'] == plot, 'Decisión'] = decision
        
        if mc:
            df.loc[(df['PLOT'] == plot) & (df['MC'] == 'VERDADERO'), 'TSG'] = tsg
    
    return df

# Cargar el dataframe (reemplazar con la carga real de datos)
# Cargar el DataFrame con datos reales
hum_tard = pd.read_excel("ENSAYO_2018-19_GDUs.xlsx", sheet_name="HUM TARD", decimal=",")

# Verificar si el DataFrame contiene la columna 'PLOT'
if 'PLOT' not in hum_tard.columns:
    raise KeyError("The column 'PLOT' is missing from the DataFrame. Please check the input data.")

# Procesar los datos
hum_tard_procesado = procesar_plots(hum_tard)


KeyError: "The column 'PLOT' is missing from the DataFrame. Please check the input data."

In [ ]:
# Convertir la columna 'PLOT' a enteros
ens_tardio['PLOT'] = ens_tardio['PLOT'].astype(int)

# Verificar el resultado
print(ens_tardio['PLOT'].head())

0    101
1    102
2    103
3    104
4    105
Name: PLOT, dtype: int32


In [ ]:
import pandas as pd

def evaluar_comportamiento(df):
    df = df.sort_values(by='Fecha')  # Ordenar por fecha para evaluar tendencias
    df['Comportamiento HM X'] = 'Esperado'
    df['Comportamiento Peso'] = 'Esperado'
    
    for i in range(1, len(df)):
        if df.iloc[i]['HUM'] > df.iloc[i-1]['HUM']:
            df.at[df.index[i], 'Comportamiento HM X'] = 'No Esperado'
        if df.iloc[i]['Peso Seco'] < df.iloc[i-1]['Peso Seco']:
            df.at[df.index[i], 'Comportamiento Peso'] = 'No Esperado'
    
    return df

def determinar_mf(df):
    df_mf = df[(df['HUM'] >= 32) & (df['HUM'] <= 40)]
    if not df_mf.empty:
        mf = df_mf.loc[df_mf['Peso Seco'].idxmax(), 'Fecha']  # Mayor peso seco dentro del rango
        return mf
    return None

def calcular_tsg(df, mf):
    df_mc = df[(df['HUM'] >= 10) & (df['HUM'] <= 16)]
    if not df_mc.empty:
        mc = df_mc.iloc[0]['Fecha']  # Primera fecha con humedad en rango MC
        hum_mf = df.loc[df['Fecha'] == mf, 'HUM'].values[0]
        hum_mc = df.loc[df['Fecha'] == mc, 'HUM'].values[0]
        tsg = (hum_mf - hum_mc) / (mc - mf).days if (mc - mf).days > 0 else None
        return mc, tsg
    return None, None

def procesar_plots(df):
    df.columns = df.columns.str.upper().str.strip()  # Convertir nombres de columnas a mayúsculas
    
    if 'PLOT' not in df.columns:
        raise KeyError("La columna 'PLOT' no está en el DataFrame. Verifique los datos de entrada.")

    df['PLOT'] = df['PLOT'].astype(int)  # Asegurar que PLOT es numérico
    df['Comportamiento Peso'] = df['Comportamiento Peso'].astype(str)  # Asegurar texto
    df['Decisión'] = df['Decisión'].astype(str)  # Asegurar texto
    df['MF'] = df['MF'].astype(bool)  # MF como VERDADERO/FALSO
    df['MC'] = df['MC'].astype(bool)  # MC como VERDADERO/FALSO
    df['TSG'] = pd.to_numeric(df['TSG'], errors='coerce')  # TSG numérico
    
    for plot in df['PLOT'].unique():
        subset = df[df['PLOT'] == plot].copy()
        subset = evaluar_comportamiento(subset)
        mf = determinar_mf(subset)
        
        if mf:
            mc, tsg = calcular_tsg(subset, mf)
            decision = 'Plot Válido'
        else:
            mc, tsg, decision = None, None, 'Se descarta'
        
        df.loc[df['PLOT'] == plot, 'Decisión'] = decision
        
        if mc:
            df.loc[(df['PLOT'] == plot) & (df['MC'] == True), 'TSG'] = tsg  # Solo asignar en filas con MC VERDADERO
    
    return df

# Cargar el DataFrame con datos reales
hum_tard = pd.read_excel("ENSAYO_2018-19_GDUs.xlsx", sheet_name="HUM TARD", decimal=",")

# Convertir nombres de columnas a mayúsculas para evitar problemas
hum_tard.columns = hum_tard.columns.str.upper().str.strip()

# Procesar los datos
hum_tard_procesado = procesar_plots(hum_tard)

# Guardar el archivo corregido
hum_tard_procesado.to_excel("HUM_TARD_PROCESADO.xlsx", index=False, decimal=",")
print("Proceso completado. Archivo guardado como 'HUM_TARD_PROCESADO.xlsx'.")


ValueError: invalid literal for int() with base 10: 'T101'

In [ ]:
import pandas as pd

# Cargar los datos de los ensayos
ens_temprano = pd.read_excel("ENSAYO_2018-2019_Tornello.xlsx", sheet_name="ENS TEMP 201819", decimal=",")
ens_tardio = pd.read_excel("ENSAYO_2018-2019_Tornello.xlsx", sheet_name="ENS TARDIO 201819", decimal=",")

# Normalizar nombres de columnas (convertir a mayúsculas y quitar espacios)
ens_temprano.columns = ens_temprano.columns.str.upper().str.strip()
ens_tardio.columns = ens_tardio.columns.str.upper().str.strip()

# Seleccionar columnas de interés y renombrar si es necesario
columnas_deseadas = [
    "AÑO", "LOCALIDAD", "ENSAYO", "PLOT", "BLOC", "IBLK", "PEDIGREE",
    "GDU_VT", "GDU_R1", "GDU_ASI", "GDU_MF", "FECHA MF", "GDU_MC", "FECHA MC",
    "TASA DE SECADO DEL GRANO", "PESO SECO MIL", "CHALAS"
]

# Crear estructura base con valores fijos
def preparar_dataframe(df, ensayo):
    df = df.copy()
    df["AÑO"] = df["PLOT"].apply(lambda x: "2018" if int(x) < 200 else "2019")  # Definir año según criterio
    df["LOCALIDAD"] = "Pergamino"  # Localidad fija
    df["ENSAYO"] = ensayo  # Tipo de ensayo ('Temprano' o 'Tardío')

    # Asegurar que todas las columnas existen en el DataFrame, aunque estén vacías
    for col in columnas_deseadas:
        if col not in df.columns:
            df[col] = None  # Crear columna vacía si no existe

    return df[columnas_deseadas]  # Devolver solo las columnas deseadas en el orden correcto

# Procesar ambos ensayos
df_temprano = preparar_dataframe(ens_temprano, "Temprano")
df_tardio = preparar_dataframe(ens_tardio, "Tardío")

# Unir ambos ensayos en un solo DataFrame
df_final = pd.concat([df_temprano, df_tardio], ignore_index=True)

# Guardar en un archivo Excel
df_final.to_excel("BASE_DATOS_UNIFICADA.xlsx", index=False)

print("✅ Proceso completado. Archivo guardado como 'BASE_DATOS_UNIFICADA.xlsx'.")


✅ Proceso completado. Archivo guardado como 'BASE_DATOS_UNIFICADA.xlsx'.
